# Week 2: Bidirectional RNN with Attention

**Notebook goal:** Implement a simple Bidirectional vanilla RNN with additive attention
for wheat futures price forecasting, following the same training framework used by
previous models in this project (TimeSeriesSplit CV + final retrain on all training data).

---

## 1  Bidirectional Vanilla RNN

A standard RNN processes $\mathbf{x}_1, \dots, \mathbf{x}_T$ left-to-right:

$$
\overrightarrow{h}_t = \tanh\!\left(W_{hh}\,\overrightarrow{h}_{t-1} + W_{xh}\,\mathbf{x}_t + b_h\right)
$$

A **bidirectional** RNN adds a second pass right-to-left:

$$
\overleftarrow{h}_t = \tanh\!\left(W_{hh}'\,\overleftarrow{h}_{t+1} + W_{xh}'\,\mathbf{x}_t + b_h'\right)
$$

At each step the two states are concatenated: $h_t = [\overrightarrow{h}_t \;\|\; \overleftarrow{h}_t] \in \mathbb{R}^{2H}$

---

## 2  Additive Attention

Instead of using only the last hidden state, attention pools all $T$ states:

$$
e_t = \mathbf{v}^\top \tanh(W_a h_t + b_a), \qquad \alpha_t = \frac{\exp(e_t)}{\sum_{t'} \exp(e_{t'})}, \qquad c = \sum_{t=1}^{T} \alpha_t h_t
$$

---

## 3  Training Strategy (matching existing models)

| Step | Description |
|---|---|
| 1. CV (`cv_train_timeseries_model`) | `TimeSeriesSplit` with 5 folds; keeps best-fold model |
| 2. Final retrain (`train_final_model_and_eval`) | Trains on **all** training data, `val_fraction=0.1` tail |
| Optuna | Tunes hyperparameters using mean CV fold RMSE as objective |

**No LSTM or GRU** — plain `tanh` RNN cells only (`nn.RNN`).

In [ ]:
# !pip install torch optuna --quiet

from abc import ABC, abstractmethod
from pathlib import Path
import warnings, pickle, copy, gc

import numpy as np
import pandas as pd
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

warnings.filterwarnings("ignore")
plt.style.use("seaborn-v0_8-whitegrid")
sns.set_context("notebook")

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device  : {DEVICE}")
print(f"PyTorch : {torch.__version__}")

## Evaluation Metrics

$$
\text{RMSE} = \sqrt{\frac{1}{n}\sum(y_t - \hat{y}_t)^2}, \quad
\text{MAE} = \frac{1}{n}\sum|y_t - \hat{y}_t|, \quad
\text{MAPE} = \frac{100}{n}\sum\left|\frac{y_t-\hat{y}_t}{y_t}\right|
$$

$$
\text{MASE} = \frac{\frac{1}{n}\sum|y_t-\hat{y}_t|}{\frac{1}{n-1}\sum|y_t-y_{t-1}|}, \quad
\text{RRMSE} = \frac{\text{RMSE}}{\bar{y}}
$$

In [ ]:
class BaseForecastModel(ABC):
    """Common interface for all weekly models."""
    def __init__(self, task_type: str, **hyperparameters):
        self.task_type = task_type
        self.hyperparameters = hyperparameters
    @abstractmethod
    def fit(self, X_train, y_train): ...
    @abstractmethod
    def predict(self, X): ...
    @abstractmethod
    def evaluate(self, X_test, y_test): ...
    @abstractmethod
    def save(self, filepath: str): ...
    @abstractmethod
    def load(self, filepath: str): ...


# ── Scaling helpers (matching existing codebase) ─────────────────────────
def fit_x_scaler_3d(X_3d: np.ndarray) -> StandardScaler:
    scaler = StandardScaler()
    scaler.fit(X_3d.reshape(-1, X_3d.shape[2]))
    return scaler

def transform_x_scaler_3d(scaler, X_3d: np.ndarray) -> np.ndarray:
    return scaler.transform(X_3d.reshape(-1, X_3d.shape[2])).reshape(X_3d.shape)

def fit_y_scaler(y: np.ndarray) -> StandardScaler:
    scaler = StandardScaler()
    scaler.fit(y.reshape(-1, 1))
    return scaler

def transform_y(scaler, y: np.ndarray) -> np.ndarray:
    return scaler.transform(y.reshape(-1, 1)).ravel()

def inverse_y(scaler, y_scaled: np.ndarray) -> np.ndarray:
    return scaler.inverse_transform(y_scaled.reshape(-1, 1)).ravel()


# ── Metrics ──────────────────────────────────────────────────────────────
def naive_persistence_forecast(y_train: np.ndarray, y_test: np.ndarray) -> np.ndarray:
    y_pred = np.empty_like(y_test, dtype=float)
    y_pred[0] = y_train[-1]
    y_pred[1:] = y_test[:-1]
    return y_pred

def regression_metrics(y_true, y_pred, y_train_for_mase) -> dict:
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    mae  = float(mean_absolute_error(y_true, y_pred))
    r2   = float(r2_score(y_true, y_pred))
    with np.errstate(divide='ignore', invalid='ignore'):
        mape = float(np.nanmean(np.abs((y_true - y_pred) / y_true)[np.isfinite(np.abs((y_true - y_pred) / y_true))]) * 100)
    naive_scale = np.mean(np.abs(np.diff(np.asarray(y_train_for_mase, dtype=float))))
    mase  = float(mae / naive_scale) if naive_scale > 0 else np.nan
    rrmse = float(rmse / np.mean(y_true)) if np.mean(y_true) != 0 else np.nan
    return {'rmse': rmse, 'mae': mae, 'mape': mape, 'mase': mase, 'rrmse': rrmse, 'r2': r2}

print("Base class + helpers ready.")

## PyTorch Architecture

### `AdditiveAttention`

Learnable soft alignment over all $T$ hidden states:
$e_t = \mathbf{v}^\top \tanh(W_a h_t)$ → softmax → weighted sum.

### `BiRNNWithAttention`

```
Input (B, T, F)
  └─ nn.RNN(bidirectional=True, nonlinearity='tanh')  →  (B, T, 2H)
       └─ AdditiveAttention                           →  (B, 2H)
            └─ Dropout + Linear                       →  (B, 1)
```

In [ ]:
class AdditiveAttention(nn.Module):
    """Bahdanau-style additive attention over a sequence of hidden states."""
    def __init__(self, hidden_size: int):
        super().__init__()
        self.W_a = nn.Linear(hidden_size, hidden_size, bias=True)
        self.v   = nn.Linear(hidden_size, 1, bias=False)

    def forward(self, H: torch.Tensor):
        # H: (B, T, hidden_size)
        scores  = self.v(torch.tanh(self.W_a(H)))   # (B, T, 1)
        weights = torch.softmax(scores, dim=1)       # (B, T, 1)
        context = (weights * H).sum(dim=1)           # (B, hidden_size)
        return context, weights.squeeze(-1)           # (B, T)


class BiRNNWithAttention(nn.Module):
    """Bidirectional vanilla RNN (tanh) with additive attention pooling."""
    def __init__(self, input_size, hidden_size=64, num_layers=2, dropout=0.2):
        super().__init__()
        self.rnn = nn.RNN(
            input_size    = input_size,
            hidden_size   = hidden_size,
            num_layers    = num_layers,
            batch_first   = True,
            bidirectional = True,
            nonlinearity  = 'tanh',
            dropout = dropout if num_layers > 1 else 0.0,
        )
        self.attention   = AdditiveAttention(hidden_size * 2)
        self.dropout_out = nn.Dropout(p=dropout)
        self.fc          = nn.Linear(hidden_size * 2, 1)

    def forward(self, x):
        H, _             = self.rnn(x)              # (B, T, 2H)
        context, weights = self.attention(H)        # (B, 2H), (B, T)
        out              = self.fc(self.dropout_out(context)).squeeze(-1)
        return out, weights


# quick smoke test
_m = BiRNNWithAttention(input_size=32, hidden_size=16, num_layers=1)
_o, _w = _m(torch.randn(4, 30, 32))
print(f"BiRNN defined — out: {tuple(_o.shape)}, attn: {tuple(_w.shape)}")
del _m, _o, _w

## Training Functions

Mirrors the Keras pattern from `AI_ML_Quants_.ipynb`:

- **`cv_train_timeseries_model`** — `TimeSeriesSplit(n_splits=5)`, trains each fold
  for a fixed `epochs`, keeps the fold with the lowest validation loss.
- **`train_final_model_and_eval`** — fits scalers on all training data, trains for
  `epochs=200` with the last `val_fraction=0.1` as a tail validation split.
- **No manual early-stopping loop** — training simply runs for `epochs` epochs,
  matching how the existing Keras models behave.

In [ ]:
def _make_loader(X, y, batch_size, shuffle):
    ds = TensorDataset(torch.tensor(X, dtype=torch.float32),
                       torch.tensor(y, dtype=torch.float32))
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle)


def _run_epoch(model, loader, criterion, optimiser=None):
    """One training or validation epoch. Returns mean loss."""
    training = optimiser is not None
    model.train() if training else model.eval()
    total, count = 0.0, 0
    ctx = torch.enable_grad() if training else torch.no_grad()
    with ctx:
        for Xb, yb in loader:
            Xb, yb = Xb.to(DEVICE), yb.to(DEVICE)
            if training:
                optimiser.zero_grad()
            pred, _ = model(Xb)
            loss = criterion(pred, yb)
            if training:
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimiser.step()
            total += loss.item() * len(Xb)
            count += len(Xb)
    return total / count


def cv_train_timeseries_model(
    X_train       : np.ndarray,
    y_train       : np.ndarray,
    X_test        : np.ndarray,
    y_test        : np.ndarray,
    hidden_size   : int   = 64,
    num_layers    : int   = 2,
    dropout       : float = 0.2,
    learning_rate : float = 1e-3,
    epochs        : int   = 100,
    batch_size    : int   = 64,
    n_splits      : int   = 5,
    plot_test     : bool  = True,
):
    """
    TimeSeriesSplit CV: trains each fold for `epochs` epochs,
    keeps the fold with lowest val loss.
    Matches the pattern in AI_ML_Quants_.ipynb (no manual early stopping).
    """
    tscv = TimeSeriesSplit(n_splits=n_splits)
    criterion = nn.MSELoss()

    best = {'val_loss': float('inf'), 'model': None,
            'x_scaler': None, 'y_scaler': None, 'fold': None}

    for fold, (tr_idx, vl_idx) in enumerate(tscv.split(X_train), start=1):
        print(f'=== Fold {fold} ===')
        torch.manual_seed(RANDOM_SEED + fold)

        X_tr, X_vl = X_train[tr_idx], X_train[vl_idx]
        y_tr, y_vl = y_train[tr_idx], y_train[vl_idx]

        # fit scalers on training split only
        x_sc = fit_x_scaler_3d(X_tr)
        y_sc = fit_y_scaler(y_tr)
        X_tr_s = transform_x_scaler_3d(x_sc, X_tr)
        X_vl_s = transform_x_scaler_3d(x_sc, X_vl)
        y_tr_s = transform_y(y_sc, y_tr)

        model = BiRNNWithAttention(
            input_size  = X_train.shape[2],
            hidden_size = hidden_size,
            num_layers  = num_layers,
            dropout     = dropout,
        ).to(DEVICE)
        opt = torch.optim.Adam(model.parameters(), lr=learning_rate)

        tr_loader = _make_loader(X_tr_s, y_tr_s, batch_size, shuffle=True)
        vl_loader = _make_loader(X_vl_s, transform_y(y_sc, y_vl), batch_size, shuffle=False)

        for ep in range(1, epochs + 1):
            _run_epoch(model, tr_loader, criterion, opt)

        fold_val_loss = _run_epoch(model, vl_loader, criterion)
        print(f'  val_loss: {fold_val_loss:.6f}')

        if fold_val_loss < best['val_loss']:
            best.update({'val_loss': fold_val_loss, 'model': copy.deepcopy(model),
                         'x_scaler': x_sc, 'y_scaler': y_sc, 'fold': fold})

    # evaluate on test using best fold's scalers
    model = best['model'].eval()
    X_test_s = transform_x_scaler_3d(best['x_scaler'], X_test)
    Xt = torch.tensor(X_test_s, dtype=torch.float32).to(DEVICE)
    with torch.no_grad():
        y_pred_sc, _ = model(Xt)
    y_pred = inverse_y(best['y_scaler'], y_pred_sc.cpu().numpy())

    metrics = regression_metrics(y_test, y_pred, y_train)
    print(f"\n=== Best fold: {best['fold']} | val_loss: {best['val_loss']:.6f} ===")
    print(f"Test RMSE : {metrics['rmse']:.6f}")
    print(f"Test R2   : {metrics['r2']:.6f}")

    if plot_test:
        plt.figure(figsize=(10, 5))
        plt.plot(y_test,  label='Actual (test)')
        plt.plot(y_pred,  label='Predicted (test)')
        plt.title('CV Best Fold — Test: Actual vs Predicted')
        plt.legend(); plt.tight_layout(); plt.show()

    return model, best['x_scaler'], best['y_scaler'], y_pred, metrics

In [ ]:
def train_final_model_and_eval(
    X_train       : np.ndarray,
    y_train       : np.ndarray,
    X_test        : np.ndarray,
    y_test        : np.ndarray,
    hidden_size   : int   = 64,
    num_layers    : int   = 2,
    dropout       : float = 0.2,
    learning_rate : float = 1e-3,
    epochs        : int   = 200,
    batch_size    : int   = 64,
    val_fraction  : float = 0.1,
    plot_test     : bool  = True,
):
    """
    Final retrain on ALL training data.
    Last `val_fraction` of training used as tail validation (no shuffle).
    Trains for a fixed `epochs` — matching the Keras pattern in AI_ML_Quants_.ipynb.
    """
    torch.manual_seed(RANDOM_SEED)

    # scalers fitted on full training set
    x_sc = fit_x_scaler_3d(X_train)
    y_sc = fit_y_scaler(y_train)
    X_tr_s  = transform_x_scaler_3d(x_sc, X_train)
    X_tst_s = transform_x_scaler_3d(x_sc, X_test)
    y_tr_s  = transform_y(y_sc, y_train)

    # tail validation split (chronological, no shuffle)
    vs = max(1, int(len(X_tr_s) * val_fraction))
    X_f, X_v = X_tr_s[:-vs], X_tr_s[-vs:]
    y_f, y_v = y_tr_s[:-vs], y_tr_s[-vs:]

    tr_loader = _make_loader(X_f, y_f, batch_size, shuffle=True)
    vl_loader = _make_loader(X_v, y_v, batch_size, shuffle=False)

    model = BiRNNWithAttention(
        input_size  = X_train.shape[2],
        hidden_size = hidden_size,
        num_layers  = num_layers,
        dropout     = dropout,
    ).to(DEVICE)
    opt       = torch.optim.Adam(model.parameters(), lr=learning_rate)
    criterion = nn.MSELoss()

    train_losses, val_losses = [], []
    for ep in tqdm(range(1, epochs + 1), desc='Final training'):
        tr_l = _run_epoch(model, tr_loader, criterion, opt)
        vl_l = _run_epoch(model, vl_loader, criterion)
        train_losses.append(tr_l)
        val_losses.append(vl_l)

    # predict on test
    model.eval()
    Xt = torch.tensor(X_tst_s, dtype=torch.float32).to(DEVICE)
    with torch.no_grad():
        y_pred_sc, _ = model(Xt)
    y_pred = inverse_y(y_sc, y_pred_sc.cpu().numpy())

    metrics = regression_metrics(y_test, y_pred, y_train)
    print('\n=== Final retrain performance ===')
    print(f"Test RMSE : {metrics['rmse']:.6f}")
    print(f"Test R2   : {metrics['r2']:.6f}")

    if plot_test:
        plt.figure(figsize=(10, 5))
        plt.plot(y_test, label='Actual (test)')
        plt.plot(y_pred, label='Predicted (test)')
        plt.title('Final Model — Actual vs Predicted (test)')
        plt.legend(); plt.tight_layout(); plt.show()

    return model, x_sc, y_sc, y_pred, metrics, train_losses, val_losses

print("Training functions ready.")

## Data Loading

Identical pipeline to previous weeks and `AI_ML_Quants_.ipynb`.

In [ ]:
BASE_DIR      = Path('/content/drive/MyDrive/Quants ')
INVESTING_DIR = BASE_DIR / 'Investing.com'
FRED_DIR      = BASE_DIR / 'FredMD_Dataset'

WHEAT_2018 = INVESTING_DIR / 'US Wheat Futures Historical Data_2018.csv'
WHEAT_2025 = INVESTING_DIR / 'US Wheat Futures Historical Data_2025.csv'
FRED_CSV   = FRED_DIR      / 'FRED.csv'

for p in [WHEAT_2018, WHEAT_2025, FRED_CSV]:
    if not p.exists(): raise FileNotFoundError(f'Missing: {p}')

FRED_FEATURES = [
    'RPI', 'W875RX1', 'CMRMTSPLx', 'IPFPNSS', 'USWTRADE', 'USTRADE',
    'BUSLOANS', 'CONSPI', 'S&P 500', 'S&P PE ratio', 'FEDFUNDS', 'TB3MS',
    'TB6MS', 'GS1', 'GS5', 'GS10', 'AAA', 'BAA', 'TB3SMFFM', 'TB6SMFFM',
    'T1YFFM', 'T5YFFM', 'T10YFFM', 'AAAFFM', 'BAAFFM', 'EXSZUSx',
    'EXJPUSx', 'EXUSUKx', 'EXCAUSx', 'PPICMM', 'UMCSENTx',
]

def load_price_csv(path):
    df = pd.read_csv(path)
    df['Date'] = pd.to_datetime(df['Date'])
    df = df.sort_values('Date').set_index('Date')
    df['Price'] = df['Price'].astype(str).str.replace(',', '', regex=False).astype(float)
    return df['Price']

def load_and_merge_wheat_prices(p18, p25):
    merged = pd.concat([load_price_csv(p18), load_price_csv(p25).iloc[1:]])
    merged = merged[~merged.index.duplicated(keep='last')].sort_index()
    merged.name = 'Price'
    return merged

def load_fred_features(path, features):
    df = pd.read_csv(path).dropna(how='all', axis=1)
    drop = [c for c in df.columns if c != 'Month'
            and (df[c].isna() | (df[c].astype(str).str.strip() == '')).any()]
    df['Month'] = pd.to_datetime(df['Month'])
    df = df.set_index('Month').sort_index().drop(columns=drop, errors='ignore')
    df = df[features].copy()
    df.index = df.index + pd.DateOffset(months=1)
    daily = pd.date_range(df.index.min(), df.index.max() + pd.offsets.MonthEnd(0), freq='D')
    return df.reindex(daily).ffill()

def build_sequences(df_all, lookback=30):
    data = df_all.values
    X, y = [], []
    for i in range(len(data) - lookback):
        X.append(data[i : i + lookback])
        y.append(data[i + lookback, -1])
    return np.array(X), np.array(y)

print(f"FRED features: {len(FRED_FEATURES)}")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

df_price    = load_and_merge_wheat_prices(WHEAT_2018, WHEAT_2025)
df_features = load_fred_features(FRED_CSV, FRED_FEATURES)
df_all = df_features.join(df_price, how='inner').dropna().sort_index()
if df_all.index.min() < pd.Timestamp('2008-01-01'):
    df_all = df_all.loc['2008-01-01':]

print("Panel shape:", df_all.shape)
print("Date range :", df_all.index.min().date(), "→", df_all.index.max().date())

lookback = 30
X, y = build_sequences(df_all, lookback=lookback)

test_size = int(X.shape[0] * 0.2)
X_train, X_test = X[:-test_size], X[-test_size:]
y_train, y_test = y[:-test_size], y[-test_size:]

print("Training set:", X_train.shape, y_train.shape)
print("Test set    :", X_test.shape,  y_test.shape)

naive_pred    = naive_persistence_forecast(y_train, y_test)
naive_metrics = regression_metrics(y_test, naive_pred, y_train)
print(f"Naive RMSE  : {naive_metrics['rmse']:.6f}")

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(13, 9), constrained_layout=True)
axes[0].plot(df_all.index, df_all['Price'], color='tab:blue')
axes[0].set_title('Wheat Futures Price (2008 – present)')
axes[0].set_ylabel('Price (USD/bushel)')

cols = ['FEDFUNDS', 'GS10', 'UMCSENTx', 'PPICMM']
cdf  = (df_all[cols] - df_all[cols].mean()) / df_all[cols].std()
for c in cols: axes[1].plot(cdf.index, cdf[c], label=c, linewidth=1.0)
axes[1].set_title('Selected Macro Features (Z-score)')
axes[1].legend(ncol=2)
plt.show()

# train/test split visualisation
split_idx  = len(y_train)
all_target = np.concatenate([y_train, y_test])
plt.figure(figsize=(13, 4))
plt.plot(all_target, color='tab:blue', label='Wheat Price')
plt.axvline(split_idx, color='red', linestyle='--', linewidth=1.5, label='Train/Test split')
plt.title('Chronological 80/20 Split'); plt.xlabel('Index'); plt.ylabel('Price')
plt.legend(); plt.tight_layout(); plt.show()

## Step 1 — Cross-Validation

Run `cv_train_timeseries_model` with default hyperparameters to sanity-check
the architecture before Optuna tuning.

In [ ]:
cv_model, cv_xs, cv_ys, cv_pred, cv_metrics = cv_train_timeseries_model(
    X_train, y_train, X_test, y_test,
    hidden_size   = 64,
    num_layers    = 2,
    dropout       = 0.2,
    learning_rate = 1e-3,
    epochs        = 100,
    batch_size    = 64,
    plot_test     = True,
)

## Step 2 — Final Retrain

Train on **all** training data (last 10% as tail validation) for 200 epochs.

In [ ]:
final_model, xs, ys, yhat, metrics, tr_losses, vl_losses = train_final_model_and_eval(
    X_train, y_train, X_test, y_test,
    hidden_size   = 64,
    num_layers    = 2,
    dropout       = 0.2,
    learning_rate = 1e-3,
    epochs        = 200,
    batch_size    = 64,
    val_fraction  = 0.1,
    plot_test     = True,
)

# loss curves
plt.figure(figsize=(10, 4))
plt.plot(tr_losses, label='Train MSE (scaled)')
plt.plot(vl_losses, label='Val MSE (scaled)')
plt.title('Final Training — Loss Curves')
plt.xlabel('Epoch'); plt.ylabel('MSE (scaled)'); plt.legend()
plt.tight_layout(); plt.show()

## Step 3 — Optuna Hyperparameter Tuning

The objective function runs `TimeSeriesSplit` CV and returns the mean fold RMSE
(in original price units), matching the Optuna pattern in `AI_ML_Quants_.ipynb`.

| Hyperparameter | Search range |
|---|---|
| `hidden_size` | {32, 64, 128} |
| `num_layers` | {1, 2} |
| `dropout` | 0.1 – 0.4 |
| `learning_rate` | 1e-5 – 1e-1 (log) |
| `epochs` | 50 – 200 |
| `batch_size` | {32, 64} |

In [ ]:
def cv_birnn_attention_rmse(
    X_train       : np.ndarray,
    y_train       : np.ndarray,
    hidden_size   : int   = 64,
    num_layers    : int   = 2,
    dropout       : float = 0.2,
    learning_rate : float = 1e-3,
    epochs        : int   = 100,
    batch_size    : int   = 64,
    n_splits      : int   = 5,
    base_seed     : int   = 1234,
) -> float:
    """Mean CV fold RMSE on original price scale — used as Optuna objective."""
    tscv = TimeSeriesSplit(n_splits=n_splits)
    criterion = nn.MSELoss()
    fold_rmses = []

    for fold, (tr_idx, vl_idx) in enumerate(tscv.split(X_train), start=1):
        torch.manual_seed(base_seed + fold)
        gc.collect()

        X_tr, X_vl = X_train[tr_idx], X_train[vl_idx]
        y_tr, y_vl = y_train[tr_idx], y_train[vl_idx]

        x_sc = fit_x_scaler_3d(X_tr);  y_sc = fit_y_scaler(y_tr)
        X_tr_s = transform_x_scaler_3d(x_sc, X_tr)
        X_vl_s = transform_x_scaler_3d(x_sc, X_vl)
        y_tr_s = transform_y(y_sc, y_tr)

        model = BiRNNWithAttention(
            input_size  = X_train.shape[2],
            hidden_size = hidden_size,
            num_layers  = num_layers,
            dropout     = dropout,
        ).to(DEVICE)
        opt = torch.optim.Adam(model.parameters(), lr=learning_rate)

        tr_loader = _make_loader(X_tr_s, y_tr_s, batch_size, shuffle=True)

        for _ in range(epochs):
            _run_epoch(model, tr_loader, criterion, opt)

        model.eval()
        Xvt = torch.tensor(X_vl_s, dtype=torch.float32).to(DEVICE)
        with torch.no_grad():
            y_pred_sc, _ = model(Xvt)
        y_pred = inverse_y(y_sc, y_pred_sc.cpu().numpy())
        fold_rmses.append(float(np.sqrt(mean_squared_error(y_vl, y_pred))))
        del model

    return float(np.mean(fold_rmses))


def objective(trial):
    hidden_size   = trial.suggest_categorical('hidden_size',   [32, 64, 128])
    num_layers    = trial.suggest_categorical('num_layers',     [1, 2])
    dropout       = trial.suggest_float('dropout',       0.1, 0.4)
    learning_rate = trial.suggest_float('learning_rate', 1e-5, 1e-1, log=True)
    epochs        = trial.suggest_int('epochs',          50,  200)
    batch_size    = trial.suggest_categorical('batch_size',     [32, 64])

    return cv_birnn_attention_rmse(
        X_train       = X_train,
        y_train       = y_train,
        hidden_size   = hidden_size,
        num_layers    = num_layers,
        dropout       = dropout,
        learning_rate = learning_rate,
        epochs        = epochs,
        batch_size    = batch_size,
    )

print("Optuna objective defined.")

In [ ]:
optuna.logging.set_verbosity(optuna.logging.INFO)

study = optuna.create_study(direction='minimize',
                            study_name='birnn_attention')
study.optimize(objective, n_trials=30, show_progress_bar=True)

print('\nBest trial:')
print(f'  RMSE : {study.best_value:.6f}')
print(f'  Params: {study.best_params}')

In [ ]:
# importance & history
try:
    from optuna.visualization.matplotlib import (
        plot_param_importances, plot_optimization_history
    )
    plot_optimization_history(study); plt.tight_layout(); plt.show()
    plot_param_importances(study);    plt.tight_layout(); plt.show()
except Exception:
    print('optuna matplotlib visualisation not available; install plotly or optuna[visualization]')

trials_df = study.trials_dataframe().sort_values('value')
display(trials_df.head(10))

## Step 4 — Final Retrain with Best Hyperparameters

Use the Optuna best params for the final model, following the same two-step
pattern as `AI_ML_Quants_.ipynb`:
1. CV to select/validate the best configuration.
2. Final retrain on all training data.

In [ ]:
bp = study.best_params

# 1) CV with best params
cv_model_opt, cv_xs_opt, cv_ys_opt, cv_pred_opt, cv_metrics_opt = cv_train_timeseries_model(
    X_train, y_train, X_test, y_test,
    hidden_size   = bp['hidden_size'],
    num_layers    = bp['num_layers'],
    dropout       = bp['dropout'],
    learning_rate = bp['learning_rate'],
    epochs        = bp['epochs'],
    batch_size    = bp['batch_size'],
    plot_test     = True,
)

# 2) Final retrain with best params
final_model_opt, xs_opt, ys_opt, yhat_opt, metrics_opt, tr_l_opt, vl_l_opt = train_final_model_and_eval(
    X_train, y_train, X_test, y_test,
    hidden_size   = bp['hidden_size'],
    num_layers    = bp['num_layers'],
    dropout       = bp['dropout'],
    learning_rate = bp['learning_rate'],
    epochs        = bp['epochs'],
    batch_size    = bp['batch_size'],
    val_fraction  = 0.1,
    plot_test     = True,
)

# loss curves for final model
plt.figure(figsize=(10, 4))
plt.plot(tr_l_opt, label='Train MSE (scaled)')
plt.plot(vl_l_opt, label='Val MSE (scaled)')
plt.title('Final Model (Optuna) — Loss Curves')
plt.xlabel('Epoch'); plt.ylabel('MSE (scaled)'); plt.legend()
plt.tight_layout(); plt.show()

In [ ]:
# ── Residual diagnostics ────────────────────────────────────────────────
resid = y_test - yhat_opt
fig, axes = plt.subplots(1, 3, figsize=(16, 4), constrained_layout=True)

axes[0].plot(resid, color='steelblue', linewidth=0.8)
axes[0].axhline(0, color='black', linestyle='--')
axes[0].set_title('Residuals over Test Period')
axes[0].set_xlabel('Test index'); axes[0].set_ylabel('Residual')

axes[1].hist(resid, bins=40, color='steelblue', edgecolor='white', alpha=0.8)
axes[1].set_title('Residual Distribution')

lo = min(y_test.min(), yhat_opt.min())
hi = max(y_test.max(), yhat_opt.max())
axes[2].scatter(y_test, yhat_opt, alpha=0.35, s=10, color='steelblue')
axes[2].plot([lo, hi], [lo, hi], 'r--', linewidth=1.5, label='Perfect fit')
axes[2].set_title('Predicted vs Actual')
axes[2].set_xlabel('Actual'); axes[2].set_ylabel('Predicted'); axes[2].legend()
plt.show()

## Attention Weight Analysis

The weights $\alpha_t$ reveal which days in the 30-day lookback the model weighted
most. Deviations from the uniform baseline $1/T$ indicate structural dependencies.

In [ ]:
final_model_opt.eval()
X_tst_s = transform_x_scaler_3d(xs_opt, X_test)
Xt = torch.tensor(X_tst_s, dtype=torch.float32).to(DEVICE)
with torch.no_grad():
    _, test_attn = final_model_opt(Xt)
test_attn = test_attn.cpu().numpy()   # (N_test, T)

mean_attn = test_attn.mean(axis=0)
std_attn  = test_attn.std(axis=0)
lags = np.arange(-lookback + 1, 1)

plt.figure(figsize=(10, 4))
plt.bar(lags, mean_attn, color='steelblue', alpha=0.8, label='Mean α')
plt.fill_between(lags, mean_attn - std_attn, mean_attn + std_attn,
                 alpha=0.3, color='steelblue', label='±1 std')
plt.axhline(1 / lookback, color='red', linestyle='--', linewidth=1,
            label=f'Uniform (1/{lookback})')
plt.title('Mean Attention Weights over Lookback Window')
plt.xlabel('Lag (days before prediction)'); plt.ylabel('α')
plt.legend(); plt.tight_layout(); plt.show()

# heatmap
N_SHOW = 100
fig, ax = plt.subplots(figsize=(14, 5))
im = ax.imshow(test_attn[:N_SHOW].T, aspect='auto', cmap='YlOrRd', origin='lower')
ax.set_yticks(range(lookback))
ax.set_yticklabels([f't-{lookback-1-i}' for i in range(lookback)], fontsize=6)
ax.set_xlabel('Test sample index'); ax.set_ylabel('Lag')
ax.set_title(f'Attention Heatmap — First {N_SHOW} Test Predictions')
plt.colorbar(im, ax=ax, fraction=0.02, pad=0.02, label='α')
plt.tight_layout(); plt.show()

In [ ]:
comparison = pd.DataFrame([
    {'model': 'Naive Persistence',           **naive_metrics},
    {'model': 'BiRNN+Attention (default CV)', **cv_metrics},
    {'model': 'BiRNN+Attention (default final)', **metrics},
    {'model': 'BiRNN+Attention (Optuna CV)',   **cv_metrics_opt},
    {'model': 'BiRNN+Attention (Optuna final)', **metrics_opt},
]).set_index('model')

print('=== Model comparison on held-out test set ===')
display(comparison.round(6))

In [ ]:
MODEL_DIR  = Path.cwd() / 'models'
MODEL_DIR.mkdir(parents=True, exist_ok=True)
MODEL_PATH = MODEL_DIR / 'week2_birnn_attention.pkl'

payload = dict(
    model_state  = final_model_opt.state_dict(),
    x_scaler     = xs_opt,
    y_scaler     = ys_opt,
    best_params  = bp,
    metrics      = metrics_opt,
    y_train_orig = y_train,
)
with open(MODEL_PATH, 'wb') as f:
    pickle.dump(payload, f)
print(f'Saved → {MODEL_PATH}')

# round-trip check
with open(MODEL_PATH, 'rb') as f:
    loaded = pickle.load(f)

chk = BiRNNWithAttention(
    input_size  = X_train.shape[2],
    hidden_size = bp['hidden_size'],
    num_layers  = bp['num_layers'],
    dropout     = bp['dropout'],
).to(DEVICE)
chk.load_state_dict(loaded['model_state'])
chk.eval()

X_tst_s2 = transform_x_scaler_3d(loaded['x_scaler'], X_test)
Xt2 = torch.tensor(X_tst_s2, dtype=torch.float32).to(DEVICE)
with torch.no_grad():
    chk_pred, _ = chk(Xt2)
chk_pred = inverse_y(loaded['y_scaler'], chk_pred.cpu().numpy())

print(f'Max |orig-loaded|: {np.max(np.abs(yhat_opt - chk_pred)):.2e}')
assert np.allclose(yhat_opt, chk_pred, atol=1e-4)
print('Save/load check passed.')